# Part 25: Professionally You — Personal AI Agent with Tool Calling

> Build a deployable AI agent that represents **you** on your personal website. It answers career questions from your LinkedIn profile, uses tools to record leads and unknown questions, sends real-time push notifications, and deploys to HuggingFace Spaces.

---

**What you'll build:**
```
User → Gradio Chat UI
          ↓
       Me-Bot (GPT + system prompt + LinkedIn PDF + tools)
          ↓              ↓
    record_user_details  record_unknown_question
          ↓              ↓
       Pushover push notification to your phone
```


## 25.1 Setup & Pushover Notifications

**Pushover** lets you send push notifications to your phone from code — free & instant.

1. Sign up at [pushover.net](https://pushover.net/)
2. Create an Application/API Token
3. Add to `.env`:
   ```
   PUSHOVER_USER=u...
   PUSHOVER_TOKEN=a...
   ```

In [ ]:
import os, json
import requests
import gradio as gr
from openai import OpenAI
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv(override=True)
openai = OpenAI()

# Pushover config
PUSHOVER_USER  = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
PUSHOVER_URL   = "https://api.pushover.net/1/messages.json"

def push(message: str):
    """Send a push notification to your phone."""
    print(f"[Push] {message}")
    if PUSHOVER_USER and PUSHOVER_TOKEN:
        requests.post(PUSHOVER_URL, data={
            "user":    PUSHOVER_USER,
            "token":   PUSHOVER_TOKEN,
            "message": message
        })

# Test it!
# push("Me-Bot is online!")

## 25.2 Tool Functions

Two tools for the agent:
- **record_user_details** — when someone shares their email, save it + notify you
- **record_unknown_question** — when the bot can't answer, log it for you to review

In [ ]:
def record_user_details(email: str, name: str = "not provided", notes: str = "not provided") -> dict:
    """Record a user who expressed interest and shared their email."""
    push(f"New lead: {name} <{email}> | Notes: {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question: str) -> dict:
    """Record a question the bot couldn't answer — so you can improve the bot."""
    push(f"Unknown question: {question}")
    return {"recorded": "ok"}

## 25.3 Tool Schema (JSON)

OpenAI needs tool definitions as JSON schema so the LLM knows when and how to call them.

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Additional context about the conversation worth recording"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
]

## 25.4 Tool Dispatcher

Two approaches — the IF statement and the elegant `globals()` approach:

In [ ]:
# Approach A: explicit IF statement (clear but verbose)
def handle_tool_calls_explicit(tool_calls):
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"Tool called: {name}")

        if name == "record_user_details":
            result = record_user_details(**args)
        elif name == "record_unknown_question":
            result = record_unknown_question(**args)
        else:
            result = {"error": f"Unknown tool: {name}"}

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})
    return results

# Approach B: elegant globals() dispatch — no IF statement needed!
def handle_tool_calls(tool_calls):
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"Tool called: {name}")
        fn = globals().get(name)                     # look up function by name
        result = fn(**args) if fn else {"error": f"{name} not found"}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})
    return results

## 25.5 Load Your Profile (LinkedIn PDF + Summary)

Place these two files in the same directory:
- `linkedin.pdf` — download from LinkedIn: Me → Settings → Data Privacy → Get a copy of your data
- `summary.txt` — a short written summary about yourself

In [ ]:
# Load LinkedIn PDF
def load_linkedin_pdf(path: str = "linkedin.pdf") -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted
    return text

# Load personal summary
def load_summary(path: str = "summary.txt") -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return "I am a passionate engineer interested in AI and technology."

NAME    = "Your Name"          # <-- change this
linkedin = load_linkedin_pdf()  # reads linkedin.pdf
summary  = load_summary()       # reads summary.txt

print(f"LinkedIn: {len(linkedin)} chars")
print(f"Summary:  {len(summary)} chars")

## 25.6 System Prompt — The Persona

In [ ]:
system_prompt = f"""You are acting as {NAME}.
You are answering questions on {NAME}'s personal website, particularly about their career, background, skills, and experience.
Your responsibility is to represent {NAME} faithfully and professionally.
Be engaging, as if talking to a potential client or future employer.

Tool usage rules:
- If you cannot answer a question, ALWAYS use record_unknown_question to log it.
- If a user shares their email, ALWAYS use record_user_details to record them.
- Try to steer interested visitors toward sharing their email.

## Summary:
{summary}

## LinkedIn Profile:
{linkedin}

With this context, chat with the user, always staying in character as {NAME}."""

print(system_prompt[:300], "...")

## 25.7 The Agent Loop (finish_reason check)

The key mechanic: keep looping while the LLM wants to call tools.

```
loop:
  call LLM
  if finish_reason == 'tool_calls' → execute tools → add results → continue loop
  if finish_reason == 'stop'       → return final answer
```

In [ ]:
def chat(message: str, history: list) -> str:
    """Me-Bot: answers as you, calls tools, loops until done."""
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            # LLM wants to call a tool
            assistant_msg = response.choices[0].message
            tool_results   = handle_tool_calls(assistant_msg.tool_calls)
            messages.append(assistant_msg)       # add assistant's tool call request
            messages.extend(tool_results)        # add tool results
            # loop again — LLM will now formulate a response based on tool output
        else:
            done = True  # finish_reason == 'stop'

    return response.choices[0].message.content

## 25.8 Launch with Gradio

In [ ]:
# Launch the Me-Bot
gr.ChatInterface(
    fn=chat,
    type="messages",
    title=f"{NAME} — Career AI",
    description=f"Chat with an AI version of {NAME}. Ask about skills, experience, or get in touch!"
).launch()

## 25.9 Deploy to HuggingFace Spaces (Free Hosting)

Turn your local Gradio app into a public URL — for free.

### Prerequisites
1. Create a free account at [huggingface.co](https://huggingface.co)
2. Go to **Settings → Access Tokens** → create a token with **WRITE** permissions
3. Add to `.env`: `HF_TOKEN=hf_xxx`

### Deploy
```bash
# Install HuggingFace CLI
uv tool install 'huggingface_hub[cli]'

# Login
hf auth login
hf auth whoami   # verify

# Deploy (run from the folder containing app.py)
uv run gradio deploy
```

When prompted:
- **App name**: `career_conversation` (or any name)
- **App file**: `app.py`
- **Hardware**: `cpu-basic` (free tier)
- **Secrets**: Add `OPENAI_API_KEY`, `PUSHOVER_USER`, `PUSHOVER_TOKEN`

### The `app.py` structure
```python
# app.py — deployed version
import os, json, requests, gradio as gr
from openai import OpenAI
from pypdf import PdfReader

# ... same code as this notebook ...

if __name__ == "__main__":
    gr.ChatInterface(fn=chat, type="messages", title="Career AI").launch()
```

### Update secrets later
1. Go to your Space on huggingface.co
2. Click the ⚙️ Settings wheel
3. Scroll to **Variables and Secrets**

## 25.10 Full `app.py` — Production Version

Complete standalone file for HuggingFace Spaces deployment:

In [ ]:
# app.py — save this as a separate file for deployment
APP_PY = '''
import os, json, requests, gradio as gr
from openai import OpenAI
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

PUSHOVER_USER  = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")

NAME = "Your Name"   # <-- change this

def push(message):
    if PUSHOVER_USER and PUSHOVER_TOKEN:
        requests.post("https://api.pushover.net/1/messages.json", data={
            "user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message
        })

def record_user_details(email, name="not provided", notes="not provided"):
    push(f"Lead: {name} <{email}> | {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Unknown Q: {question}")
    return {"recorded": "ok"}

tools = [
    {"type": "function", "function": {
        "name": "record_user_details",
        "description": "Record a user who shared their email address.",
        "parameters": {"type": "object",
            "properties": {"email": {"type": "string"}, "name": {"type": "string"}, "notes": {"type": "string"}},
            "required": ["email"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "record_unknown_question",
        "description": "Record a question that could not be answered.",
        "parameters": {"type": "object",
            "properties": {"question": {"type": "string"}},
            "required": ["question"], "additionalProperties": False}
    }},
]

def handle_tool_calls(tool_calls):
    results = []
    for tc in tool_calls:
        fn = globals().get(tc.function.name)
        result = fn(**json.loads(tc.function.arguments)) if fn else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})
    return results

# Load profile
reader = PdfReader("linkedin.pdf")
linkedin = "".join(p.extract_text() or "" for p in reader.pages)
with open("summary.txt") as f:
    summary = f.read()

system_prompt = f"""You are acting as {NAME} on their personal website.
Be professional and engaging. Use tools to record leads and unknown questions.
## Summary:\n{summary}\n## LinkedIn:\n{linkedin}"""

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        if resp.choices[0].finish_reason == "tool_calls":
            msg = resp.choices[0].message
            messages.append(msg)
            messages.extend(handle_tool_calls(msg.tool_calls))
        else:
            done = True
    return resp.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages", title=f"{NAME} — Career AI").launch()
'''

# Write app.py
with open("app.py", "w") as f:
    f.write(APP_PY.strip())
print("app.py written — ready for: uv run gradio deploy")

## 25.11 Summary

| Component | Purpose |
|-----------|--------|
| `pypdf` | Load LinkedIn profile PDF |
| `summary.txt` | Personal background context |
| System prompt | Persona — LLM acts as you |
| `record_user_details` | Tool to capture email leads |
| `record_unknown_question` | Tool to log knowledge gaps |
| `handle_tool_calls()` | Dispatcher using `globals()` |
| Agent loop | `while not done` + `finish_reason` check |
| Pushover | Real-time phone notifications |
| Gradio | `ChatInterface` for web UI |
| HuggingFace Spaces | Free public hosting via `gradio deploy` |

### Extensions
- Add a **RAG knowledge base** from your blog posts or projects
- Add the **Generator-Evaluator** loop from Part 20 for quality control
- Add a **SQL tool** to query a Q&A database
- Add a **calendar tool** to let visitors book a meeting

---

Return to [Index](index.ipynb) | Previous: [Part 24 — MCP](Part24_Model_Context_Protocol.ipynb)